In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from data_loader import *

c:\Users\lcj20\AppData\Local\Programs\Python\Python310\lib\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# Load Different Versions of MovieLens

In [3]:
# Website of movielens:
# https://grouplens.org/datasets/movielens/
# Download link for movielens dataset:
# https://files.grouplens.org/datasets/movielens/

#A = load_movielen('ml-25m', rd_seed=3175)
#A = load_movielen('ml-latest-small', rd_seed=3175)
A = load_movielen_100k('ml-100k')
#A = load_movielen_1m('ml-1m')
#A = load_movielen_10m('ml-10m')

m,n = A.shape

assert(isinstance(A, np.ndarray) or isinstance(A, sp.csr_matrix))
sparse = not isinstance(A, np.ndarray)

maxk = min(m,n)
if m*n > 1<<24 or sparse:
    maxk = min(200, min(m,n)-1) # sp.svds requres 0 < k < min(m,n) (not <= min(m,n))

print(f'----- Metadata of A -----')
print('A is stored as '+('sparse matrix(sp.csr_matrix).' if sparse else 'dense matrix(np.ndarray).'))
print(f'm = {m}, n = {n}')
print(f'k is taken from 1 to {maxk}')

----- Metadata of A -----
A is stored as dense matrix(np.ndarray).
m = 943, n = 1682
k is taken from 1 to 943


# Do SVD

In [4]:
print('Doing SVD...')
if sparse:
    U,  S,  Vh  = sp.linalg.svds(A, k=400, tol=0.1) # find first 400 ranks
    # scipy.sparse.linalg.svds returns singular values in ascending order
    S = S[::-1]
    U = U[:,::-1]
    Vh= Vh[::-1,:]
else:
    U,  S,  Vh  = np.linalg.svd(A)
V = Vh.T.conj()

Doing SVD...


# Find $k$
Specifically, $k=\min\{k' : \ |T - T_{\le k'}|^2 \le \varepsilon_{\rm approx}\,|T|^2 \}$

In [6]:
ε_approx = 0.5
if sparse:
    ratio = np.cumsum(S**2)/sp.linalg.norm(A, ord='fro')**2
else:
    ratio = np.cumsum(S**2)/np.linalg.norm(A, ord='fro')**2
for i in range(1, ratio.shape[0]):
    if ratio[i-1]<ε_approx and ratio[i]>=ε_approx: cutoff_k = i
print('k =', cutoff_k)

k = 23


# Compute concrete value of $(\varepsilon, \delta)$
Parameters:

$C_{\varepsilon}=C_\delta=2$: extra factor for robustness

`Trowmin` - minimal row norm $\min_i |T_i|$

In [10]:
def eps_delta(μ, m, n, k, ε_approx, Trowmin, C_eps=2, C_delta=2):
    eps = C_eps * np.sqrt(μ*k / n)

    fac1 = 1/min(m,n)
    fac2 = 1/np.sqrt(m*n)
    delta_nom = (μ*k)**2 * ((fac1 + fac2)**2 - fac1**2)
    delta_den = Trowmin*ε_approx
    delta = delta_nom/delta_den
    delta = C_delta * delta
    return eps, delta

ε, δ = eps_delta(μ=9, m=m, n=n, k=cutoff_k, ε_approx=0.5, Trowmin=20, C_eps=2, C_delta=2)
print(f'(ε, δ) = ({ε:.2f}, {δ:.5f})')

(ε, δ) = (0.70, 0.01983)
